In [1]:
import numpy as np
import pandas as pd

from preprocessing_utils import ensure_nltk_resources, preprocess_texts
ensure_nltk_resources()

from metrics_utils import *

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize, LabelEncoder

from sklearn.cluster import KMeans, AgglomerativeClustering
import skfuzzy as fuzz

seed = 42
np.random.seed(seed)

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/bernardod/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/bernardod/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
df = pd.read_json("../datasets/fixed_dataset.json")

def get_first_label(category: str) -> str:
    return str(category).split(",")[0].strip()

df["category"] = df["category"].apply(get_first_label)
label_encoder = LabelEncoder()
y_true_encoded = label_encoder.fit_transform(df["category"].apply(get_first_label))

In [3]:
CONFIG = {
    # Datasets can have different column names
    "title_column": "title",
    "abstract_column": "abstract",

    # We can change this in the future for BERT for example
    "vectorizer": "tfidf",
    
    "tfidf": dict(min_df=3, max_df=0.7, max_features=3000, ngram_range=(1, 3)),
    
    "use_svd": True,
    "svd_components": [20, 25, 30, 35, 50, 75],  # grid
    
    "K_values": [2, 3, 4, 6, 8, 10, 12, 15],

    "algorithms": ["fcm", "kmeans", "agglomerative"],

}

In [4]:
def tfidf_vectorize(texts):
    vectorizer = TfidfVectorizer(
        ngram_range=CONFIG["tfidf"]["ngram_range"],
        stop_words="english",
        min_df=CONFIG["tfidf"]["min_df"],
        max_df=CONFIG["tfidf"]["max_df"],
        max_features=CONFIG["tfidf"]["max_features"],
    )
    return vectorizer.fit_transform(texts)

In [5]:
texts = (df[CONFIG["title_column"]] + " " + df[CONFIG["abstract_column"]]).tolist()
texts_clean = preprocess_texts(texts, use_pos=True)

VECTORIZERS = {
    "tfidf": tfidf_vectorize,
}

X_base = VECTORIZERS[CONFIG["vectorizer"]](texts_clean)

In [6]:
def build_X(X_base, n_comp=None, use_svd=True):
    """
    Returns final X that all algorithms will use.
    - If use_svd=False: returns dense array (Assumes it's already normalized).
    - Se use_svd=True: apply SVD with n_comp.
    """
    if use_svd:
        svd = TruncatedSVD(n_components=n_comp, random_state=seed)
        X = svd.fit_transform(X_base)
        X = normalize(X, norm="l2")
    else:
        X = X_base.toarray() if hasattr(X_base, "toarray") else np.asarray(X_base)

    return X

In [7]:
def cluster_fcm(X, K, m=1.7, error=0.005, maxiter=1000):
    cntr, U, *_rest, fpc = fuzz.cluster.cmeans(
        X.T, c=K, m=m, error=error, maxiter=maxiter,
        metric="cosine", seed=seed
    )
    U = U.T
    labels = U.argmax(axis=1)
    return labels, {"fpc": float(fpc), "U": U}

def cluster_kmeans(X, K, max_iter=300):
    model = KMeans(n_clusters=K, random_state=seed, init="k-means++", max_iter=max_iter)
    labels = model.fit_predict(X)
    return labels, {"inertia": float(model.inertia_)}

def cluster_agglomerative_cluster(X, K):
    model = AgglomerativeClustering(n_clusters=K, metric="cosine", linkage="average")
    labels = model.fit_predict(X)
    return labels


In [8]:
# For cmeans only
def diagnose_collapse(U):
    eps = 1e-12
    K = U.shape[1]
    entropy = -np.sum(U * np.log(U + eps), axis=1)
    entropy_norm = float(np.mean(entropy) / np.log(K))
    avg_max_memb = float(U.max(axis=1).mean())
    return {
        "collapsed": entropy_norm > 0.85,
        "entropy_norm": entropy_norm,
        "avg_max_memb": avg_max_memb,
    }

def evaluate_all(X, y_true, y_pred):
    return {
        "ARI": calculate_ari(y_true, y_pred),
        "NMI": calculate_nmi(y_true, y_pred),
        "ACC": calculate_accuracy(y_true, y_pred),
        "SIL": calculate_silhouette(X, y_pred)
    }

In [9]:
def append_result(alg_name, y_pred, extras):
    extras = extras or {}
    metrics = evaluate_all(X, y_true_encoded, y_pred)
    row = {**base_row, "alg": alg_name, **metrics, **extras}
    rows.append(row)


rows = []

if CONFIG["use_svd"]:
    n_comp_list = CONFIG["svd_components"]
else:
    n_comp_list = [None]

for n_comp in n_comp_list:
    X = build_X(X_base, n_comp=n_comp, use_svd=CONFIG["use_svd"])

    for K in CONFIG["K_values"]:

        base_row = {
            "vectorizer": CONFIG["vectorizer"],
            "use_svd": CONFIG["use_svd"],
            "n_comp": n_comp,
            "K": K,
            "collapsed": False,
            "entropy_norm": np.nan,
            "avg_max_memb": np.nan,
        }
        
        if "fcm" in CONFIG["algorithms"]:
            y_pred, extra = cluster_fcm(X, K)

            collapse = diagnose_collapse(extra["U"])
            extras = {
                **collapse,
                "fpc": extra["fpc"],
            }
            append_result("FCM", y_pred, extras)
            
        if "kmeans" in CONFIG["algorithms"]:
            y_pred, extra = cluster_kmeans(X, K)
            append_result("KMeans", y_pred, {
                "inertia": extra["inertia"]
            })

        if "agglomerative" in CONFIG["algorithms"]:
            y_pred = cluster_agglomerative_cluster(X, K)
            append_result("AGG", y_pred, None)

results_all = pd.DataFrame(rows)
results_all.reset_index(drop=True, inplace=True)

In [10]:
filtered = results_all.copy()
filtered = filtered[~(filtered["collapsed"] == True)]

top_per_alg = (
    filtered
    .sort_values(by=["ARI", "NMI", "ACC"], ascending=False)
)

top_per_alg.head(40)

,vectorizer,use_svd,n_comp,K,collapsed,entropy_norm,avg_max_memb,alg,ARI,NMI,ACC,SIL,fpc,inertia
59,tfidf,True,30,6,False,NaN,NaN,AGG,0.113669,0.260735,0.404959,0.102274,NaN,NaN
29,tfidf,True,25,3,False,NaN,NaN,AGG,0.099417,0.225078,0.396694,0.119169,NaN,NaN
41,tfidf,True,25,10,False,NaN,NaN,AGG,0.098572,0.271592,0.347107,0.153892,NaN,NaN
35,tfidf,True,25,6,False,NaN,NaN,AGG,0.097626,0.245596,0.396694,0.088091,NaN,NaN
32,tfidf,True,25,4,False,NaN,NaN,AGG,0.095715,0.217883,0.380165,0.106147,NaN,NaN
83,tfidf,True,35,6,False,NaN,NaN,AGG,0.093929,0.253495,0.396694,0.102251,NaN,NaN
62,tfidf,True,30,8,False,NaN,NaN,AGG,0.090052,0.260923,0.388430,0.124201,NaN,NaN
8,tfidf,True,20,4,False,NaN,NaN,AGG,0.088538,0.239199,0.404959,0.126110,NaN,NaN
11,tfidf,True,20,6,False,NaN,NaN,AGG,0.086697,0.227482,0.363636,0.147182,NaN,NaN
80,tfidf,True,35,4,False,NaN,NaN,AGG,0.084270,0.215754,0.404959,0.096446,NaN,NaN


In [11]:
filtered = results_all.query("K == 8")
filtered = filtered[~(filtered["collapsed"] == True)]

filtered.sort_values(["ARI", "NMI", "ACC"], ascending=False)

,vectorizer,use_svd,n_comp,K,collapsed,entropy_norm,avg_max_memb,alg,ARI,NMI,ACC,SIL,fpc,inertia
62,tfidf,True,30,8,False,NaN,NaN,AGG,0.090052,0.260923,0.388430,0.124201,NaN,NaN
38,tfidf,True,25,8,False,NaN,NaN,AGG,0.082879,0.244985,0.355372,0.133497,NaN,NaN
86,tfidf,True,35,8,False,NaN,NaN,AGG,0.079312,0.245607,0.347107,0.130471,NaN,NaN
14,tfidf,True,20,8,False,NaN,NaN,AGG,0.071673,0.228268,0.338843,0.168104,NaN,NaN
110,tfidf,True,50,8,False,NaN,NaN,AGG,0.063102,0.234582,0.330579,0.061274,NaN,NaN
61,tfidf,True,30,8,False,NaN,NaN,KMeans,0.048713,0.230109,0.347107,0.120625,NaN,76.708048
134,tfidf,True,75,8,False,NaN,NaN,AGG,0.046461,0.233792,0.322314,0.068422,NaN,NaN
60,tfidf,True,30,8,False,0.721314,0.470731,FCM,0.042894,0.220525,0.305785,0.120805,0.364368,NaN
12,tfidf,True,20,8,False,0.640365,0.550019,FCM,0.042280,0.202948,0.297521,0.171989,0.429979,NaN
109,tfidf,True,50,8,False,NaN,NaN,KMeans,0.040335,0.198631,0.322314,0.090462,NaN,86.329077
